This problem finetune a gpt2 model for summarization using the SAMSum dataset which was used in lecture07-t5.ipynb

We will prepare appropriate prompts, and finetune gpt2 using reference responses in the SAMSum dataset, similar to what's done in lecture07-t5.ipynb, and similar to gpt2 classification example in lecture08-gpt2.ipynb


Import necessary libraries

In [1]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset, Dataset
from tqdm import tqdm 
import torch
import numpy as np
import random


# reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'device={device}')


device=cuda


Load gpt2 model and tokenizer and generate text

In [2]:
model_name = "gpt2"  # You can use "gpt2", "gpt2-medium", "gpt2-large", or "gpt2-xl"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Use the end-of-sequence token as the padding token
tokenizer.pad_token_id = tokenizer.eos_token_id  # Set the padding token ID to match the EOS token ID

model = GPT2LMHeadModel.from_pretrained(model_name).to(device)

/opt/conda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Test Zeroshot summarization

In [3]:
# Function to perform summarization using GPT-2
def gpt2_summarize(example):
    # Construct the prompt
    instruction='Please summarize the above dialog: '
    prompt = f"Diaglog: {example['dialogue']}\n {instruction}"


    # Encode the input text
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # Generate the output
    output = model.generate(
        input_ids,
        max_length=len(input_ids[0]) + 128,  # Limiting output length to prevent lengthy completions
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.pad_token_id,
        top_k=50,
        top_p=0.95,
        temperature=0
    )

    # Decode the output to text
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract the sentiment from the generated text
    summary = generated_text.split(instruction)[-1]
    return summary


def print_example(example):
    generated_summary= gpt2_summarize(example)
    print(f"Input Dialogue: {example['dialogue']}")
    print(f"Reference Summary: {example['summary']}")
    print(f"Generated Summary: {generated_summary}")
    
    
print("Loading the SAMSum dataset...")
datasets = load_dataset('samsum')

# Display a sample from the dataset
print("Sample Zero-shot Performance:")

print_example(datasets['test'][0])


Loading the SAMSum dataset...
Sample Zero-shot Performance:


/opt/conda/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Input Dialogue: Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye
Reference Summary: Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
Generated Summary:  Hanna: What's up? Amanda? Hannah? Amanda: Oh, I'm sorry. I just wanted to say that I was really happy to see you. Am I right? I mean, you're a really nice person. Hannah, what's wrong?  
I'm not sure what to do.   
What's going on?    Hannah is still in the hospital.   Amandla: Is she okay? Is her head okay, or is she just fine? 
Is she fine, amanda.  Amandi: She's fine. She just needs to get some rest.


Now, we can see zeroshot gpt2 doesn't summarize well. 

We want to prepare tokenized_dataset for finetuning the gpt2 model for summarization.

Please look at lecture08-gpt2.ipynb and modify the code  to implement generate_prompt_response(example) function

- prompt should be consistent with what's defined in the  gpt2_summarize function  above. 
- response should come from the reference summary in the 'summary' column. 

In [4]:
# Implement generate_prompt_response(example)  function
def generate_prompt_response(example):
    
    # prompt = f": {review_text}\n Is the review's sentiment positive or negative?"
    prompt_response_text = f"Given a dialogue: {example['dialogue']}, this is the summary: {example['summary']}"
    return {"formatted_text": prompt_response_text}


We now use generate_prompt_response() function to tokenize the dataset

We also evaluate the gpt2 model on test data prior to finetuning

In [5]:
# Tokenize the data
def tokenize_function(examples):
    # print(examples.keys())
    return tokenizer(examples["formatted_text"], padding="max_length", truncation=True, max_length=256)

# Tokenize the dataset
train_dataset = datasets['train'].map(generate_prompt_response).map(tokenize_function, batched=True)
test_dataset = datasets['test'].map(generate_prompt_response).map(tokenize_function, batched=True)

 
def evaluate_model(message):
    eval_args = TrainingArguments(
        output_dir="./results",     
        per_device_eval_batch_size=8,
        logging_dir="./logs",       
        eval_strategy="no"  
    )
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    evaluator = Trainer(
        model=model,                
        args=eval_args,         
        eval_dataset=test_dataset,  
        data_collator=data_collator,
    )

    eval_results =evaluator.evaluate()
    print(f"Evaluation Results {message}: loss={eval_results['eval_loss']:.2f}")
    

evaluate_model('prior to finetuning')

Evaluation Results prior to finetuning: loss=4.28


We can now finetune 'model' using tokenized_dataset.

Please see lecture08-gpt2.ipynb and lecture07-t5.ipynb

Make sure that 
- you use train_dataset to finetune the model.
- you use test_dataset to evaluate the model.
- perform evaluate after each epoch
- try to reasonably tune your training parameters

In [7]:
# Implement gpt2 model finetuning
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    logging_steps=100,
    eval_strategy = 'epoch',  
    learning_rate=1e-4,
    weight_decay=0.01,
    seed=seed,
)
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset = test_dataset,
    tokenizer=tokenizer,
)

# Fine-tune the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.553200,2.453357
2,2.419400,2.418997
3,2.353900,2.409724


TrainOutput(global_step=693, training_loss=2.4737645048771757, metrics={'train_runtime': 1009.5167, 'train_samples_per_second': 43.779, 'train_steps_per_second': 0.686, 'total_flos': 5774031323136000.0, 'train_loss': 2.4737645048771757, 'epoch': 3.0})

We now test the model you trained

In [8]:
evaluate_model('after finetuning')

print("\nGenerating example summaries...")
test_samples = datasets['test'].select(range(3))  # Select a few examples to display
for i, example in enumerate(test_samples):
    print(f"\nExample {i+1}:")    
    print_example(example)


Evaluation Results after finetuning: loss=2.41

Generating example summaries...

Example 1:


/opt/conda/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Input Dialogue: Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye
Reference Summary: Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
Generated Summary:  Hannah, Amanda and Larry are at Betty and Lemmy's.   They are going to meet at Larry's place. They will text Betty. She will not text them. Amanda will. Larry will call her. He will be at her place in an hour. Hannah will write him. , this is the summary: Amanda is not sure if Betty has Betty his number. It is Larry who called Betty last year. The conversation is about the phone call. Betty will send him the text. Amanda will tell him to text her an